In [1]:
!pip install -q "surya-ocr==0.14.7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 46.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 77.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 27.0 MB/s eta 0:00:00
ERROR: pip's de

In [2]:
import html
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

In [3]:
# ---------------------------------------------------------------------------
# Base
# ---------------------------------------------------------------------------
def _to_pil(image) -> Image.Image:
    """Accept a PIL image, a filesystem path, or a numpy array."""
    if isinstance(image, Image.Image):
        return image
    if isinstance(image, (str, Path, os.PathLike)):
        img = Image.open(image)
        img.load()                      # decode now; the file handle is released
        return img
    if hasattr(image, "__array_interface__") or hasattr(image, "shape"):
        return Image.fromarray(image)
    raise TypeError(f"cannot interpret {type(image).__name__} as an image")


class OCREngine:
    """Base class. Implement `_transcribe`; the contract is handled here."""

    name = "base"
    device_kind = "cpu"

    def transcribe(self, images):
        """image | list[image] -> str | list[str]. See the module docstring."""
        single = isinstance(images, (Image.Image, str, Path, os.PathLike)) or not (
            isinstance(images, (list, tuple)))
        batch = [images] if single else list(images)
        if not batch:
            return [] if not single else ""

        texts = self._transcribe([_to_pil(im) for im in batch])

        # Normalise whatever the backend gave us to exactly one string per input.
        if len(texts) != len(batch):
            raise RuntimeError(
                f"{self.name}: got {len(texts)} texts for {len(batch)} images")
        texts = ["" if t is None else str(t) for t in texts]
        return texts[0] if single else texts

    # Callable form, so an engine can be passed anywhere a function is expected.
    __call__ = transcribe

    def _transcribe(self, images: list[Image.Image]) -> list[str]:
        raise NotImplementedError

    def close(self) -> None:
        pass

In [4]:
class SuryaEngine(OCREngine):
    """Surya 1 (surya-ocr 0.14.x), recognition only.

    Detection is skipped the way Surya supports natively: pass `bboxes` covering the
    whole crop and `det_predictor=None`. Same reasoning as PaddleOCREngine -- the input
    is already one line.

    Surya is a DOCUMENT model, so two things must be undone before its output is
    comparable to a line recogniser's, and both cost a lot of accuracy if ignored.
    Measured over the first 40 benchmark lines (akshara error rate, lower is better):

        math_mode=False, strip tags, longest segment   0.187   <- defaults here
        math_mode=False, strip tags, first segment     0.246
        math_mode=False, strip tags, join segments     0.300
        math_mode=False, raw output                    0.413
        math_mode=True,  raw output                    0.523

    1. MARKUP. Surya emits inline HTML -- <br>, <b>, <i>, and with math_mode=True a lot
       of <math> -- which is formatting, not a reading error. Tags are stripped and
       entities unescaped. math_mode=False also makes the underlying reading better
       here, not just less marked up.
    Batches are kept small and MPS memory is returned after each one (see `_free`):
    a 1044-line pass segfaulted at batch 19 of 33 without it.

    2. THE NEIGHBOURING LINE. These crops carry a sliver of the line above or below,
       and a document model reads it, returning two <br>-separated segments (6 of those
       40). Which segment is the real line varies -- sometimes first, sometimes second
       -- so `take="longest"` picks the longest, the full line being longer than a
       partial sliver. It is a heuristic, but a reference-free one: it never looks at
       the ground truth. "join" (score everything Surya read) and "first" are available
       for comparison; the table above is the argument for the default.
    """

    name = "surya"
    device_kind = "gpu"

    _TAG = re.compile(r"<[^>]+>")
    _BR = re.compile(r"<br\s*/?>", re.IGNORECASE)

    def __init__(self, math_mode: bool = False, take: str = "longest",
                 batch_size: int = 16, recognition_batch_size: int | None = None,
                 free_every_batch: bool = True):
        from surya.common.surya.schema import TaskNames
        from surya.recognition import RecognitionPredictor

        if take not in ("longest", "first", "join"):
            raise ValueError(f"take must be longest|first|join, got {take!r}")
        self._rec = RecognitionPredictor()
        self._task = TaskNames.ocr_with_boxes
        self.math_mode = math_mode
        self.take = take
        self.batch_size = batch_size
        self.recognition_batch_size = recognition_batch_size
        self.free_every_batch = free_every_batch
        self.n_failed = 0
        self.n_multi_segment = 0

    @staticmethod
    def _free():
        """Hand MPS memory back between batches.

        Surya segfaulted partway through a 1044-line pass (batch 19 of 33) with
        everything else already released, which is the signature of memory growing
        across batches rather than of a conflict with another engine. torch does not
        return MPS blocks on its own, so they are dropped explicitly here.
        """
        import gc

        gc.collect()
        try:
            import torch

            if torch.backends.mps.is_available():
                torch.mps.empty_cache()
            elif torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    def _segments(self, text: str) -> list[str]:
        text = self._BR.sub("\n", str(text))
        text = self._TAG.sub("", text)
        text = html.unescape(text)
        return [seg.strip() for seg in text.split("\n") if seg.strip()]

    def _pick(self, text: str) -> str:
        segs = self._segments(text)
        if not segs:
            return ""
        if len(segs) > 1:
            self.n_multi_segment += 1
        if self.take == "first":
            return segs[0]
        if self.take == "join":
            return " ".join(segs)
        return max(segs, key=len)

    def _transcribe(self, images: list[Image.Image]) -> list[str]:
        out: list[str] = []
        for i in range(0, len(images), self.batch_size):
            chunk = [im.convert("RGB") for im in images[i:i + self.batch_size]]
            # One box per image covering all of it: "read this whole crop as one line".
            bboxes = [[[0, 0, im.width, im.height]] for im in chunk]
            try:
                results = self._rec(
                    chunk, task_names=[self._task] * len(chunk), bboxes=bboxes,
                    det_predictor=None, math_mode=self.math_mode,
                    recognition_batch_size=self.recognition_batch_size)
            except Exception as exc:
                print(f"[{self.name}] batch of {len(chunk)} failed: {exc}")
                self.n_failed += len(chunk)
                out.extend([""] * len(chunk))
                continue
            for res in results:
                lines = getattr(res, "text_lines", None) or []
                out.append(self._pick(" ".join(ln.text for ln in lines)))
            if self.free_every_batch:
                self._free()
        return out


In [5]:
# ---------------------------------------------------------------------------
# Registry — add an engine here and the benchmark can name it.
# ---------------------------------------------------------------------------
def build_engine(spec: dict) -> OCREngine:
    """{"engine": "tesseract", ...kwargs} -> a ready OCREngine."""
    spec = dict(spec)
    kind = spec.pop("engine")
    builders = {
        # "tesseract": TesseractEngine,
        # "paddle": PaddleOCREngine,
        "surya": SuryaEngine,
        # "model": TeluguOCREngine,
    }
    if kind not in builders:
        raise ValueError(f"unknown engine {kind!r}; have {sorted(builders)}")
    return builders[kind](**spec)

In [6]:
import unicodedata
from collections import Counter
from dataclasses import dataclass, field

import regex

EMPTY = "∅"                      # stands in for "nothing" in a confusion pair
_GRAPHEME = regex.compile(r"\X")


def edit_distance(a: str, b: str) -> int:
    m, n = len(a), len(b)
    if m == 0:
        return n
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[n]


# ---------------------------------------------------------------------------
# Tokenisation
# ---------------------------------------------------------------------------
def normalize(text) -> str:
    """NFC + collapsed whitespace: compare characters, not spacing conventions."""
    return " ".join(unicodedata.normalize("NFC", str(text)).split())


def to_chars(text) -> list[str]:
    return list(str(text))


def to_aksharas(text) -> list[str]:
    """Unicode grapheme clusters — the same split the model's tokenizer uses."""
    return _GRAPHEME.findall(str(text))


def to_words(text) -> list[str]:
    return str(text).split()


# ---------------------------------------------------------------------------
# Alignment
# ---------------------------------------------------------------------------
def align(ref: list, hyp: list) -> list[tuple[str, str, str]]:
    """Levenshtein alignment of hyp against ref.

    Returns the edit script as (op, ref_token, hyp_token) with op in
    equal/sub/del/ins. `del` = present in ref, missing from hyp; `ins` = present in
    hyp, absent from ref. Directionality is what makes the breakdown meaningful, so it
    is fixed here rather than left to the caller.

    The distance implied (sub + ins + del) is the ordinary Levenshtein distance, so
    error rates computed from this agree exactly with test_model.edit_distance.
    """
    m, n = len(ref), len(hyp)
    # dp[i][j] = distance between ref[:i] and hyp[:j]
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        dp[i][0] = i
    for j in range(1, n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        ri = ref[i - 1]
        for j in range(1, n + 1):
            if ri == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1],   # substitute
                                   dp[i - 1][j],       # delete from ref
                                   dp[i][j - 1])       # insert into hyp
    ops: list[tuple[str, str, str]] = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1] and dp[i][j] == dp[i - 1][j - 1]:
            ops.append(("equal", ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            ops.append(("sub", ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append(("del", ref[i - 1], EMPTY))
            i -= 1
        else:
            ops.append(("ins", EMPTY, hyp[j - 1]))
            j -= 1
    ops.reverse()
    return ops


# ---------------------------------------------------------------------------
# Accumulators
# ---------------------------------------------------------------------------
@dataclass
class LevelStats:
    """Corpus-wide totals for one tokenisation level."""

    unit: str
    sub: int = 0
    ins: int = 0
    dele: int = 0
    ref_len: int = 0
    exact: int = 0
    n: int = 0
    confusions: Counter = field(default_factory=Counter)

    @property
    def errors(self) -> int:
        return self.sub + self.ins + self.dele

    @property
    def error_rate(self) -> float:
        return self.errors / max(self.ref_len, 1)

    @property
    def exact_rate(self) -> float:
        return self.exact / max(self.n, 1)

    def update(self, ref_tokens: list, hyp_tokens: list, collect_confusions=True):
        ops = align(ref_tokens, hyp_tokens)
        for op, r, h in ops:
            if op == "equal":
                continue
            if op == "sub":
                self.sub += 1
            elif op == "ins":
                self.ins += 1
            else:
                self.dele += 1
            if collect_confusions:
                self.confusions[(r, h)] += 1
        self.ref_len += max(len(ref_tokens), 1)
        self.exact += int(ref_tokens == hyp_tokens)
        self.n += 1
        return ops

    def as_dict(self) -> dict:
        return {
            "unit": self.unit,
            "error_rate": self.error_rate,
            "exact": self.exact_rate,
            "sub": self.sub,
            "ins": self.ins,
            "del": self.dele,
            "errors": self.errors,
            "ref_len": self.ref_len,
        }

    def top_confusions(self, k=15) -> list[dict]:
        return [{"ref": r, "hyp": h, "count": c,
                 "kind": "del" if h == EMPTY else "ins" if r == EMPTY else "sub"}
                for (r, h), c in self.confusions.most_common(k)]


def score_pairs(predictions, references, collect_confusions=True) -> dict:
    """Score one engine's output. -> {"char": LevelStats, "akshara": ..., "word": ...}"""
    levels = {
        "char": (LevelStats("char"), to_chars),
        "akshara": (LevelStats("akshara"), to_aksharas),
        "word": (LevelStats("word"), to_words),
    }
    for hyp, ref in zip(predictions, references):
        hyp_n, ref_n = normalize(hyp), normalize(ref)
        for name, (stats, tokenize) in levels.items():
            # Confusions over words are unbounded and rarely actionable; the useful
            # ones are aksharas (what to train on) and characters (what to normalise).
            stats.update(tokenize(ref_n), tokenize(hyp_n),
                         collect_confusions=collect_confusions and name != "word")
    return {name: stats for name, (stats, _) in levels.items()}

In [7]:
import gc
import json
import os
import random
import time
from collections import Counter

from datasets import load_dataset

In [8]:
import gc
import json
import os
import random
import time
from collections import Counter

from datasets import load_dataset


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_bench(repo, split, token=None):
    """The published benchmark split from the Hub.

    Deduplication, empty-text removal and the human-verified filter all happened when
    the dataset was built (data_curation/wikisource/push_eval_to_hub.py), so nothing is
    re-derived here -- the published rows ARE the reference set. The guards in
    select_samples remain only for things a consumer must still decide, like whether a
    given engine can encode a given image.
    """
    ds = load_dataset(repo, split=split, token=token)
    print(f"[data] {repo} (split={split})")
    print(f"[data] {len(ds)} rows, columns: {ds.column_names}")
    return ds


def select_samples(ds, n_samples, seed, max_source_width,
                   text_field="text", id_field="line_image"):
    """-> (images, references, keys), at most n_samples of them.

    Rows are walked in a seeded shuffle and decoded lazily, stopping once n_samples are
    in hand, so a small run does not decode the whole split. n_samples=None walks the
    split in order and takes everything.

    A row no engine could fairly be scored on is skipped for ALL engines -- images too
    wide for the model's encoder would otherwise be scored as empty for it alone.
    """
    order = list(range(len(ds)))
    if n_samples is not None:
        random.Random(seed).shuffle(order)

    images, references, keys = [], [], []
    dropped = Counter()
    for i in order:
        if n_samples is not None and len(images) >= n_samples:
            break
        row = ds[i]
        text = str(row.get(text_field, "")).strip()
        if not text:
            dropped["empty reference"] += 1
            continue
        img = row["image"]
        if max_source_width and img.width / max(img.height, 1) * 64 > max_source_width:
            dropped["too wide for the encoder"] += 1
            continue
        images.append(img)
        references.append(text)
        keys.append(row.get(id_field) or f"row-{i}")

    for reason, count in dropped.items():
        print(f"[data] dropped {count} ({reason})")
    scope = f"{len(images)} sampled (seed {seed})" if n_samples is not None \
        else f"all {len(images)}"
    print(f"[data] scoring {scope} of {len(ds)} rows")
    return images, references, keys


# ---------------------------------------------------------------------------
# Engine lifecycle
#
# Engines are built one at a time and released before the next one is constructed.
# They bring incompatible native runtimes -- torch/MPS, paddle, tesseract, surya --
# and holding them all at once segfaulted the full-split run: two copies of our
# 325MB model sat on MPS alongside paddle's and surya's, and surya died mid-batch.
# Nothing here is needed for a single-engine run; it is the price of comparing them
# in one process.
# ---------------------------------------------------------------------------
def preflight(engine_specs):
    """Missing files named in the specs, checked before anything heavy is built.

    Building lazily costs the old fail-fast behaviour: a typo'd checkpoint used to
    surface at startup and would now surface after the earlier engines had run. A
    path check is cheap and catches that same mistake.
    """
    missing = []
    for spec in engine_specs:
        for key in ("checkpoint", "vocab_file"):
            path = spec.get(key)
            if path and not os.path.exists(path):
                missing.append(f"{spec.get('engine', '?')}.{key}: {path}")
    return missing


def max_source_width(engine_specs):
    """Widest image the engines can take, without constructing them.

    Only our model has a limit, and it is a declared config value rather than
    something learned from the checkpoint, so the spec is enough.
    """
    widths = [spec.get("max_image_width", 2048)
              for spec in engine_specs if spec.get("engine") == "model"]
    return max(widths, default=0)


def release(engine):
    """Drop an engine and give its device memory back."""
    del engine
    gc.collect()
    try:
        import torch

        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------
def score(predictions, references):
    """Error rates at three units, the edit-op breakdown, and confusion pairs.

    `cer_raw` is kept alongside the normalized rates because it is the number
    src/encoder_decoder/test_model.py prints (whitespace-stripped, no NFC), so past
    eval_results.json runs stay comparable. Everything else is on normalized text.
    """
    levels = score_pairs(predictions, references)

    raw_edits = raw_ref = empty = 0
    for hyp, ref in zip(predictions, references):
        hyp_s, ref_s = str(hyp).strip(), str(ref).strip()
        raw_edits += edit_distance(hyp_s, ref_s)
        raw_ref += max(len(ref_s), 1)
        empty += int(not hyp_s)

    return {
        "n": len(references),
        "cer_raw": raw_edits / max(raw_ref, 1),
        "cer": levels["char"].error_rate,
        "aer": levels["akshara"].error_rate,
        "wer": levels["word"].error_rate,
        "exact": levels["char"].exact_rate,
        "empty_outputs": empty,
        "levels": {name: st.as_dict() for name, st in levels.items()},
        "confusions": {
            "akshara": levels["akshara"].top_confusions(25),
            "char": levels["char"].top_confusions(25),
        },
    }


# ---------------------------------------------------------------------------
# Reporting
# ---------------------------------------------------------------------------
def print_table(results):
    """Error rates, then WHERE the errors come from (sub/ins/del at akshara level)."""
    order = sorted(results, key=lambda k: results[k]["summary"]["aer"])
    w = max(len(k) for k in results) + 2
    width = w + 74

    print()
    print("=" * width)
    print(f"{'engine':<{w}}{'CER':>8}{'AER':>8}{'WER':>8}{'exact':>8}"
          f"{'sub':>7}{'ins':>7}{'del':>7}{'empty':>7}{'sec/line':>10}")
    print("=" * width)
    for name in order:
        s = results[name]["summary"]
        ak = s["levels"]["akshara"]
        print(f"{name:<{w}}{s['cer']:>8.4f}{s['aer']:>8.4f}{s['wer']:>8.4f}"
              f"{s['exact']:>8.3f}{ak['sub']:>7}{ak['ins']:>7}{ak['del']:>7}"
              f"{s['empty_outputs']:>7}{results[name]['seconds_per_line']:>10.3f}")
    print("=" * width)
    any_sum = next(iter(results.values()))["summary"]["levels"]
    print(f"lower is better. CER=code points, AER=aksharas, WER=words; sub/ins/del "
          f"are akshara counts over {any_sum['akshara']['ref_len']} reference aksharas "
          f"({any_sum['char']['ref_len']} chars, {any_sum['word']['ref_len']} words).")
    print("ranked by AER — one wrong syllable counts once, which is how Telugu reads.")


def _show(token):
    """Render a token so whitespace is visible.

    Space confusions are among the most common and the most actionable -- dropped
    spaces are the single biggest source of our model's deletions -- but printed raw
    they look like a blank and get read as noise.
    """
    return "␣" if token == " " else token.replace(" ", "␣")


def print_confusions(results, unit="akshara", k=10):
    """The aksharas each engine reliably gets wrong — what to target with more data."""
    print(f"\n--- top {k} {unit} confusions per engine  (ref -> hyp; "
          f"{EMPTY} = nothing, ␣ = space) ---")
    for name in sorted(results, key=lambda x: results[x]["summary"]["aer"]):
        rows = results[name]["summary"]["confusions"][unit][:k]
        if not rows:
            continue
        cells = [f"{_show(r['ref'])}->{_show(r['hyp'])} ({r['count']})" for r in rows]
        print(f"\n  {name}")
        for i in range(0, len(cells), 5):
            print("    " + "   ".join(f"{c:<16}" for c in cells[i:i + 5]).rstrip())


def print_examples(results, references, keys, k=3):
    """Same lines across every engine — the quickest way to see how failures differ."""
    if k <= 0 or not references:
        return
    names = list(results)
    print(f"\n--- {min(k, len(references))} sample lines ---")
    for i in range(min(k, len(references))):
        print(f"\n[{keys[i]}]")
        print(f"  {'reference':<16} {references[i]!r}")
        for name in names:
            print(f"  {name:<16} {results[name]['predictions'][i]!r}")


def main(dataset_repo, split, engine_specs, n_samples, seed, out_json,
         token=None, image_field="line_image", text_field="text",
         print_k=3, confusion_k=10):
    ds = load_bench(dataset_repo, split, token=token)
    if len(ds) == 0:
        raise SystemExit(f"{dataset_repo} split={split} is empty")

    missing = preflight(engine_specs)
    if missing:
        raise SystemExit("missing files referenced by ENGINES:\n  " +
                         "\n  ".join(missing))

    images, references, keys = select_samples(
        ds, n_samples, seed, max_source_width(engine_specs),
        text_field=text_field, id_field=image_field)
    if not images:
        raise SystemExit("no usable samples")

    results, unavailable = {}, {}
    for spec in engine_specs:
        label = spec.get("engine", "?")
        # Construct, run, then release before the next engine is built. Holding all
        # of them at once segfaulted: two copies of our 325MB model on MPS, plus
        # paddle's and surya's runtimes, and surya died mid-batch on the full split.
        try:
            engine = build_engine(spec)
        except Exception as exc:
            unavailable[label] = f"{type(exc).__name__}: {exc}"
            print(f"[setup] SKIPPING {label}: {type(exc).__name__}: {exc}")
            continue

        print(f"\n[run] {engine.name} over {len(images)} lines ...")
        t0 = time.time()
        predictions = engine(images)
        elapsed = time.time() - t0

        summary = score(predictions, references)
        results[engine.name] = {
            "summary": summary,
            "seconds": elapsed,
            "seconds_per_line": elapsed / max(len(images), 1),
            "predictions": predictions,
            "n_failed": getattr(engine, "n_failed", 0),
        }
        print(f"[run] {engine.name}: CER {summary['cer']:.4f}  "
              f"AER {summary['aer']:.4f}  WER {summary['wer']:.4f}  "
              f"exact {summary['exact']:.3f}  ({elapsed:.1f}s)")
        engine.close()
        release(engine)

    if not results:
        raise SystemExit("no engine produced any results")

    print_table(results)
    print_confusions(results, unit="akshara", k=confusion_k)
    print_examples(results, references, keys, k=print_k)
    if unavailable:
        print("\nnot benchmarked:")
        for name, why in unavailable.items():
            print(f"  {name}: {why}")

    if out_json:
        payload = {
            "config": {
                "dataset": dataset_repo, "split": split,
                "n_samples": n_samples, "seed": seed,
                "engines": engine_specs, "unavailable": unavailable,
            },
            "summary": {k: v["summary"] | {"seconds_per_line": v["seconds_per_line"]}
                        for k, v in results.items()},
            "samples": [
                {"line_image": keys[i], "reference": references[i],
                 **{name: results[name]["predictions"][i] for name in results}}
                for i in range(len(references))
            ],
        }
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        print(f"\nper-sample results -> {out_json}")

    return results

In [ ]:
DATASET_REPO = "harsha-desaraju/telugu-line-ocr-bench"
SPLIT = "test"
HF_TOKEN = None       # None uses the cached login / HF_TOKEN env var; public repo

VOCAB_FILE = f"/kaggle/input/datasets/harshadesaraju1999/telugu-tokenizer-vocab/telugu-vocab.json"
CKPT = (f"/models/encoder_decoder/results_stage_2_mid/telugu-ocr-stage2/"
        f"checkpoint-34000/model.safetensors")

N_SAMPLES = None        # None = the whole split; an int subsamples it (seeded)
SEED = 42
OUT_JSON = f"/kaggle/working/benchmark_results.json"
PRINT_K = 3           # sample lines printed side by side
CONFUSION_K = 10      # top akshara confusions printed per engine

# Add an engine here; the rest of the file does not change.
ENGINES = [
    # psm 13 + `tel` alone, not the repo's usual psm 7 + `tel+eng` — see
    # TesseractEngine's docstring for the measurements behind that.
    # {"engine": "tesseract", "lang": "tel", "psm": 13, "upscale": 2.0},
    # {"engine": "paddle", "lang": "te"},
    # math_mode off, longest <br>-segment — see SuryaEngine for the measurements.
    {"engine": "surya", "math_mode": False, "take": "longest"},
    # {"engine": "model", "checkpoint": CKPT, "vocab_file": VOCAB_FILE,
    #  "decode": "ctc"},
    # {"engine": "model", "checkpoint": CKPT, "vocab_file": VOCAB_FILE,
    #  "decode": "joint", "lam": 0.3, "beam_width": 5},
]
# -----------------------------------------------------------------------

main(DATASET_REPO, SPLIT, ENGINES, N_SAMPLES, SEED, OUT_JSON,
     token=HF_TOKEN, print_k=PRINT_K, confusion_k=CONFUSION_K)

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1044 [00:00<?, ? examples/s]

[data] harsha-desaraju/telugu-line-ocr-bench (split=test)
[data] 1044 rows, columns: ['image', 'text', 'n_graphemes', 'line_image', 'slug', 'source_file']
[data] scoring all 1044 of 1044 rows



[run] surya over 1044 lines ...


Recognizing Text:  69%|██████▉   | 11/16 [00:01<00:00, 10.10it/s]